In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from matplotlib import pyplot as plt
import os
from tqdm import tqdm 

In [2]:
catalogue_amphibians = pd.read_csv(r'../results/area-of-habitat/catalogue-amphibians.csv')
catalogue_birds = pd.read_csv(r'../results/area-of-habitat/catalogue-birds.csv')
catalogue_mammals = pd.read_csv(r'../results/area-of-habitat/catalogue-mammals.csv')
catalogue_reptiles = pd.read_csv(r'../results/area-of-habitat/catalogue-reptiles.csv')

In [3]:
country_raster_1km_ = rio.open(r"../results/country-rasters/country-raster-eck4-1km.tif")

In [ ]:
def make_rarity_map(taxon, indicator, category=None, res='1km'):
    country_raster_ = rio.open(f"../results/country-rasters/country-raster-eck4-{res}.tif")
    richness_map = np.full(country_raster_.shape, fill_value=0).astype(np.float64)
    catalogue = pd.read_csv(os.path.join('..', 'results', f'area-of-habitat-{res}', f'catalogue-{taxon}.csv'))
    if category is None:
        the_slice = catalogue[catalogue['have_a_AOH'] == True]
    else:
        the_slice = catalogue[(catalogue['have_a_AOH'] == True) & (catalogue['category'] == category)]

    for i in tqdm(the_slice.index):
        a_species = catalogue.loc[i, 'name']
        a_path = os.path.join('..', 'results', f'area-of-habitat-{res}', taxon, f'{a_species}.tif')
        src = rio.open(a_path)
        a_AOH = src.read(1)
        [row_min, row_max, col_min, col_max] = catalogue.loc[i, ['row_min', 'row_max', 'col_min', 'col_max']]

        if indicator == 'species':
            richness_map[row_min:row_max+1, col_min:col_max+1] += a_AOH 
        elif indicator == 'rarity':
            a_sum = np.sum(a_AOH)
            richness_map[row_min:row_max+1, col_min:col_max+1] += (a_AOH / a_sum)

    if indicator == 'rarity':
        print(np.sum(richness_map))

    if category is None:
        output_path = os.path.join('..', 'results', f'area-of-habitat-{res}', f'rarity-weighted-richness-{taxon}-{res}.tif')
    else:
        output_path = os.path.join('..', 'results', f'area-of-habitat-{res}', f'rarity-weighted-richness-{taxon}-{category}-{res}.tif')
        
    with rio.open(
        output_path,
        'w', 
        driver='GTiff', 
        height=country_raster_.shape[0], 
        width=country_raster_.shape[1], 
        count=1, 
        dtype=richness_map.dtype, 
        crs=country_raster_.crs, 
        transform=country_raster_.transform, 
        compress='lzw'
    ) as dst:
        dst.write(richness_map, 1)

In [ ]:
make_rarity_map('amphibians', 'rarity')

100%|██████████| 6215/6215 [00:33<00:00, 183.18it/s]


6214.999999999972


In [ ]:
make_rarity_map('mammals', 'rarity')

100%|██████████| 5237/5237 [02:43<00:00, 32.05it/s]


5236.999999999984


In [ ]:
make_rarity_map('reptiles', 'rarity')

100%|██████████| 7948/7948 [02:07<00:00, 62.29it/s] 


7948.000000000038


In [ ]:
make_rarity_map('birds', 'rarity')

100%|██████████| 9927/9927 [08:01<00:00, 20.61it/s] 


9927.000000000084


In [ ]:
make_rarity_map('amphibians', 'rarity', category='CR')
make_rarity_map('amphibians', 'rarity', category='EN')
make_rarity_map('amphibians', 'rarity', category='VU')
make_rarity_map('amphibians', 'rarity', category='NT')

100%|██████████| 331/331 [00:00<00:00, 502.57it/s]


330.9999999999954


In [ ]:
make_rarity_map('mammals', 'rarity', category='CR')
make_rarity_map('mammals', 'rarity', category='EN')
make_rarity_map('mammals', 'rarity', category='VU')
make_rarity_map('mammals', 'rarity', category='NT')

100%|██████████| 172/172 [00:00<00:00, 230.53it/s]


172.00000000000162


100%|██████████| 451/451 [00:03<00:00, 141.55it/s]


451.0000000000076


100%|██████████| 504/504 [00:06<00:00, 80.82it/s] 


504.0000000000017


100%|██████████| 354/354 [00:06<00:00, 56.90it/s]


353.9999999999986


In [ ]:
make_rarity_map('reptiles', 'rarity', category='CR')
make_rarity_map('reptiles', 'rarity', category='EN')
make_rarity_map('reptiles', 'rarity', category='VU')
make_rarity_map('reptiles', 'rarity', category='NT')

100%|██████████| 183/183 [00:02<00:00, 90.47it/s] 


182.999999999992


100%|██████████| 557/557 [00:03<00:00, 185.01it/s]


557.00000000001


100%|██████████| 424/424 [00:03<00:00, 127.50it/s]


423.9999999999954


100%|██████████| 442/442 [00:01<00:00, 242.59it/s]


441.999999999991


In [ ]:
make_rarity_map('birds', 'rarity', category='CR')
make_rarity_map('birds', 'rarity', category='EN')
make_rarity_map('birds', 'rarity', category='VU')
make_rarity_map('birds', 'rarity', category='NT')

100%|██████████| 133/133 [00:02<00:00, 57.22it/s]


133.00000000000102


100%|██████████| 297/297 [00:06<00:00, 43.72it/s]


296.9999999999827


100%|██████████| 559/559 [00:19<00:00, 28.31it/s] 


559.0000000000005


100%|██████████| 831/831 [00:22<00:00, 36.95it/s] 


831.0000000000023
